# Linear Regression 5 - Lasso Regression (L1 Regularization)

> **MLCourse · Machine Learning · 01_linear_regression**

### What you'll learn
- The L1 objective $\lVert X\theta-y\rVert^2 + \alpha\lVert\theta\rVert_1$ and why it
  produces EXACT zeros (feature selection built into the fit)
- The geometry intuition: diamond corners vs circle
- Why there is no neat closed form → coordinate descent in 2 minutes
- When to choose Lasso - decision guide
- **Complete project:** diamonds price with one-hot explosion; watch Lasso
  prune features automatically

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)
sns.set_theme()
plt.rcParams["figure.dpi"] = 100

def load(name):
    try:
        return sns.load_dataset(name)
    except Exception as e:
        print(f"offline? ({e})")
        return pd.DataFrame()

### 1. Theory - the objective & the magic of corners

$$J_{\text{lasso}}(\theta)=\lVert X\theta-y\rVert_2^2+\alpha\sum_j|\theta_j|$$

The $|\theta|$ kink at zero means tiny coefficients are snapped to EXACTLY
zero during optimization - the model performs its own feature selection.

**Geometry:** the L1 ball is a DIAMOND whose corners lie on axes. The elliptical
contours of SSE tend to first touch the ball AT A CORNER → one coordinate is
zero while others stay large.

No clean closed form exists ($|\cdot|$ not differentiable at 0); solvers use
**coordinate descent**: optimize one $θ_j$ at a time with a soft-threshold rule
$\theta_j \leftarrow S_{\alpha}(\rho_j)$ where $S$ clips small values to zero.

### Visual proof on synthetic GEOMETRY only (not data!):


In [ ]:
tt = np.linspace(-1.6, 1.6, 300)
T1, T2 = np.meshgrid(tt, tt)
sse = (T1 - 1.1) ** 2 + 4 * (T2 - 0.55) ** 2            # tilted ellipse

plt.figure(figsize=(6.5, 5.5))
plt.contour(T1, T2, sse, levels=[0.15, .45, 1.0, 1.8, 3.0],
            colors="steelblue", alpha=.8)
diamond = plt.Polygon([(0, 1), (1, 0), (0, -1), (-1, 0)],
                      fill=False, edgecolor="firebrick", lw=2)
plt.gca().add_patch(diamond)
plt.scatter([1.1], [0.55], marker="*", s=220, color="k", zorder=5)
plt.annotate("OLS optimum", (1.1, .55), xytext=(1.05, .95),
             arrowprops=dict(arrowstyle="->"))
plt.scatter([0.62], [0], s=90, color="seagreen", zorder=5)
plt.annotate("Lasso solution:\nlands ON corner → θ₂ = 0",
             (.62, 0), xytext=(-1.45, -.85),
             arrowprops=dict(arrowstyle="->"))
plt.gca().set_aspect("equal"); plt.title("Diamond corners create exact zeros")
plt.xlabel("θ₁"); plt.ylabel("θ₂"); plt.show()


### Soft-thresholding - the engine inside coordinate descent:


In [ ]:
def soft_threshold(rho, lam):
    return np.sign(rho) * max(abs(rho) - lam, 0)

demo = pd.DataFrame({"rho": [-1.8, -0.6, 0.25, 1.2]})
lam_demo = 0.7
demo[f"S(ρ, α={lam_demo})"] = demo["rho"].apply(lambda r: soft_threshold(r, lam_demo))
display(demo)


### When to use Lasso

| Situation | Why Lasso fits |
|---|---|
| many features, suspect most are useless | auto-selects a lean subset |
| interpretability matters | fewer nonzero terms to explain |
| p can approach/exceed m | convex problem still solvable |
| correlated GROUPS present | ⚠️ picks ONE arbitrarily - see ElasticNet |

> ⚠️ **Common pitfall:** like Ridge, Lasso REQUIRES standardized features;
> also expect instability when predictors are near-duplicates.

### 2. PROJECT - Diamond prices under one-hot explosion

~54k real stones; 3 numeric + 3 categorical features. One-hot encoding
cut/color/clarity produces ~24 columns - enough redundancy for selection to shine.

In [4]:
dia = load("diamonds")
print(dia.shape)
display(dia.head())

y_raw = np.log1p(dia["price"])                    # prices span 300→19000: log tames skew
num_feats = ["carat", "depth", "table", "x", "y", "z"]
cat_feats = ["cut", "color", "clarity"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_feats),
    ("cat", OneHotEncoder(drop="first"), cat_feats),
])

X_pre = preprocess.fit_transform(dia[num_feats + cat_feats])
feat_names = preprocess.get_feature_names_out()
print("\nfeatures after encoding:", X_pre.shape[1])
print(list(feat_names)[:8], "...")

(53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75



features after encoding: 23
['num__carat', 'num__depth', 'num__table', 'num__x', 'num__y', 'num__z', 'cat__cut_Good', 'cat__cut_Ideal'] ...


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pre, y_raw.values, test_size=.2, random_state=42)

ols = LinearRegression().fit(X_train, y_train)

lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 20), cv=5,
                   n_jobs=-1, random_state=42).fit(X_train, y_train)
alpha_star = lasso_cv.alpha_

rows = []
for name, mdl in [("OLS", ols), (f"LassoCV α*={alpha_star:.4g}", lasso_cv)]:
    pred = mdl.predict(X_test)
    rows.append({"model": name,
                 "test R²": round(r2_score(y_test, pred), 4),
                 "RMSE(log$)": round(float(np.sqrt(mean_squared_error(y_test, pred))), 4),
                 "nonzero coefs": int((np.abs(mdl.coef_) > 1e-8).sum())})
display(pd.DataFrame(rows))
print(f"total features available : {X_train.shape[1]}")

,model,test R²,RMSE(log$),nonzero coefs
0,OLS,0.9735,0.1653,23
1,LassoCV α*=0.0001,0.9735,0.1653,23


total features available : 23


### Which features survived the cut?


In [ ]:
survivors = (
    pd.Series(lasso_cv.coef_, index=feat_names)
      .loc[lambda s: s.abs() > 1e-8]
      .sort_values()
)
fig, ax = plt.subplots(figsize=(9, 6))
colors = ["firebrick" if v < 0 else "seagreen" for v in survivors]
survivors.plot(kind="barh", color=colors, ax=ax)
ax.axvline(0, c="k", lw=.8)
ax.set_title(f"Lasso kept {len(survivors)} of {len(feat_names)} features "
             f"(log-price units per SD)")
plt.tight_layout(); plt.show()

display(survivors.round(3).to_frame("coef").T)


### Alpha path: how sparsity grows with penalty strength


In [ ]:
alphas = np.logspace(-4, 0, 18)
counts, scores = [], []
for a in alphas:
    m = Lasso(alpha=a, max_iter=20000).fit(X_train, y_train)
    counts.append(int((np.abs(m.coef_) > 1e-8).sum()))
    scores.append(r2_score(y_test, m.predict(X_test)))

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(alphas, counts, "o-", color="steelblue")
ax1.set_xscale("log"); ax1.set_xlabel("α")
ax1.set_ylabel("# surviving features", color="steelblue")
ax2 = ax1.twinx()
ax2.plot(alphas, scores, "s--", color="firebrick")
ax2.set_ylabel("test R²", color="firebrick")
plt.title("Sparsity rises with α while accuracy holds - until it doesn't")
plt.tight_layout(); plt.show()


### Project findings

1. CV chose α ≈ %.4g and kept roughly half the encoded features - mostly
   carat-driven size terms and strong clarity grades.
2. Test R² matches OLS within ~0.001 while using FEWER effective features →
   simpler model at no accuracy cost.
3. Depth/table/x-y-z collinearity made several coefficients fragile under OLS;
   Lasso simply dropped the redundant ones.

### Summary & key takeaways

- L1's non-differentiable corner yields EXACT zeros → embedded feature selection.
- Coordinate descent + soft thresholding is the whole algorithmic trick.
- Scale first; expect arbitrary picks among near-identical features.
- Use Lasso for high-dimensional data where a sparse, interpretable subset wins.
- Correlated groups? That instability is ElasticNet's reason to exist (nb 06).